# LycoS-Unicas-IDS2018 → Parquet Splits (Rosay et al. Protocol)

Converts `LycoS18_dataset.csv` into the same three-file Parquet format used by LycoS-IDS2017,
following the **exact splitting methodology** of Rosay et al. (ICISSP 2022 / WI-IAT 2021).

```
data/lycos_unicas_ids2018/
├── train_set.parquet      (50% of each attack class + equal benign)
├── crossval_set.parquet   (25% of each attack class + equal benign)
└── test_set.parquet       (25% of each attack class + equal benign)
```

### Rosay et al. splitting protocol (quoted verbatim)
> *"The basic idea is to build sets by randomly selecting 50% of each attack for the training set,
> 25% for the cross-validation set and 25% for the test set, making sure that each instance is
> used just once. Then, each set is completed with randomly selected instances of normal traffic,
> without replacement. The result is a balanced dataset in terms of normal traffic versus attacks,
> but still imbalanced in terms of attack types."*

Key properties:
- **Per-class splitting** of attack flows — each class independently split 50/25/25
- **Benign flows** randomly sampled to match the total attack count in each split
- **No replacement** — every flow appears in exactly one split
- `crossval_set` and `test_set` are the **same size** (both 25% of attacks + equal benign)
- Only `train_set` + `crossval_set` are used during training; `test_set` is held out entirely

Reference: Rosay et al., Table 2 (ICISSP 2022): benign — 220,316 / 110,158 / 110,158; bot — 367 / 183 / 183

## 1. Imports & configuration

In [ ]:
import os
import warnings
import numpy as np
import pandas as pd
from IPython.display import display, Markdown

warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', 30)
pd.set_option('display.float_format', '{:.4f}'.format)

# ── Configuration ─────────────────────────────────────────────────────────────
CSV_PATH     = 'LycoS-Unicas-IDS2018.csv'         # path to raw CSV
OUT_DIR      = 'data/lycos_unicas_ids2018'    # output folder
LABEL_COL    = 'Label'                        # label column in the CSV
BENIGN_LABEL = 'Benign'                       # exact string for benign class

# Rosay et al. (ICISSP 2022) split fractions — applied per attack class
TRAIN_FRAC   = 0.50
CV_FRAC      = 0.25
TEST_FRAC    = 0.25   # remainder after floor rounding, so cv == test size

RANDOM_STATE = 42
CHUNK_SIZE   = 500_000

os.makedirs(OUT_DIR, exist_ok=True)
print(f'Output directory : {OUT_DIR}')
print(f'Split fractions  : {TRAIN_FRAC:.0%} train / '
      f'{CV_FRAC:.0%} crossval / {TEST_FRAC:.0%} test')
print('Splitting method : per attack class (Rosay et al.), '
      'benign matched to attack count without replacement')

Output directory : data/lycos_unicas_ids2018
Split fractions  : 50% train / 25% crossval / 25% test
Splitting method : per attack class (Rosay et al.), benign matched to attack count without replacement


## 2. Step 1 — Chunked CSV → clean intermediate Parquet

Reads in `CHUNK_SIZE`-row chunks (avoids loading 5.1 GB at once).  
Per chunk: strip whitespace, replace `Inf`/`-Inf`/`'Infinity'` → `NaN` → drop,  
downcast `float64 → float32` and `int64 → int32`.  
**Idempotent** — skips if clean Parquet already exists.

In [ ]:
import pyarrow as pa
import pyarrow.parquet as pq

CLEAN_PARQUET = os.path.join(OUT_DIR, '_lycos2018_clean.parquet')

if os.path.exists(CLEAN_PARQUET):
    print(f'Clean Parquet already exists:\n  {CLEAN_PARQUET}')
    print('Delete it to force re-conversion.')
else:
    writer = None
    total_rows = dropped_rows = 0

    for i, chunk in enumerate(pd.read_csv(
            CSV_PATH,
            chunksize  = CHUNK_SIZE,
            low_memory = False,
            na_values  = ['Infinity', 'infinity', 'inf', '-inf', ''],
    )):
        chunk.columns = chunk.columns.str.strip()
        if LABEL_COL in chunk.columns:
            chunk[LABEL_COL] = chunk[LABEL_COL].str.strip()

        chunk.replace([np.inf, -np.inf], np.nan, inplace=True)
        before = len(chunk)
        chunk.dropna(inplace=True)
        dropped_rows += before - len(chunk)
        if len(chunk) == 0:
            continue

        for col in chunk.select_dtypes('float64').columns:
            chunk[col] = chunk[col].astype('float32')
        for col in chunk.select_dtypes('int64').columns:
            chunk[col] = chunk[col].astype('int32')

        table = pa.Table.from_pandas(chunk, preserve_index=False)
        if writer is None:
            writer = pq.ParquetWriter(CLEAN_PARQUET, table.schema,
                                      compression='snappy')
        writer.write_table(table)
        total_rows += len(chunk)

        if (i + 1) % 5 == 0:
            print(f'  Chunks: {i+1:3d} | Kept: {total_rows:>11,} | '
                  f'Dropped: {dropped_rows:>8,}')

    if writer:
        writer.close()
    size_mb = os.path.getsize(CLEAN_PARQUET) / 1e6
    print(f'\nDone. Kept: {total_rows:,} | Dropped: {dropped_rows:,} | '
          f'Size: {size_mb:.1f} MB')

  Chunks:   5 | Kept:   2,500,000 | Dropped:        0
  Chunks:  10 | Kept:   5,000,000 | Dropped:        0
  Chunks:  15 | Kept:   7,500,000 | Dropped:        0
  Chunks:  20 | Kept:  10,000,000 | Dropped:        0
  Chunks:  25 | Kept:  12,500,000 | Dropped:        0

Done. Kept: 13,691,268 | Dropped: 0 | Size: 1409.3 MB


## 3. Step 2 — Load cleaned Parquet and inspect

In [ ]:
df = pd.read_parquet(CLEAN_PARQUET)
print(f'Loaded : {df.shape[0]:,} rows  ×  {df.shape[1]} columns')
print(f'RAM    : {df.memory_usage(deep=True).sum() / 1e9:.2f} GB\n')

# Normalise label column
if LABEL_COL not in df.columns:
    matches = [c for c in df.columns if c.lower() == LABEL_COL.lower()]
    if matches:
        df.rename(columns={matches[0]: LABEL_COL}, inplace=True)
    else:
        raise ValueError(f'Label column "{LABEL_COL}" not found. '
                         f'Columns: {list(df.columns)[:10]}')

benign_mask = df[LABEL_COL] == BENIGN_LABEL
df_benign   = df[benign_mask].copy()
df_attacks  = df[~benign_mask].copy()

print(f'Benign  : {len(df_benign):>10,}  ({len(df_benign)/len(df)*100:.1f}%)')
print(f'Attacks : {len(df_attacks):>10,}  ({len(df_attacks)/len(df)*100:.1f}%)')
print(f'Attack classes: {df_attacks[LABEL_COL].nunique()}\n')

counts = df_attacks[LABEL_COL].value_counts()
display(pd.DataFrame({
    'attack_count': counts,
    'train_50%':  (counts * TRAIN_FRAC).astype(int),
    'cv_25%':     (counts * CV_FRAC).astype(int),
    'test_25%':   counts - (counts * TRAIN_FRAC).astype(int)
                               - (counts * CV_FRAC).astype(int),
}).rename_axis(LABEL_COL))

Loaded : 13,691,268 rows  ×  78 columns
RAM    : 5.09 GB

Benign  : 10,000,000  (73.0%)
Attacks :  3,691,268  (27.0%)
Attack classes: 13



,attack_count,train_50%,cv_25%,test_25%
Label,,,,
DoS Hulk,1802966,901483,450741,450742
DDoS HOIC,1074379,537189,268594,268596
DDoS LOIC-HTTP,289328,144664,72332,72332
FTP-Patator,190300,95150,47575,47575
DoS Slowhttptest,105550,52775,26387,26388
Bot,96154,48077,24038,24039
SSH-Patator,92648,46324,23162,23162
DoS GoldenEye,26861,13430,6715,6716
DoS Slowloris,10274,5137,2568,2569


## 4. Step 3 — Per-class attack splitting (Rosay et al. exact method)

For **each attack class independently**:
1. Shuffle all instances with a fixed seed
2. Assign: first 50% → `train_set`, next 25% → `crossval_set`, remainder → `test_set`
3. Every instance used exactly once (no replacement)

Using `floor()` for train and cv boundaries means `test` absorbs any rounding remainder,
guaranteeing crossval ≤ test rather than losing samples.

In [ ]:
attack_train_parts, attack_cv_parts, attack_test_parts = [], [], []
split_log = []

for cls in sorted(df_attacks[LABEL_COL].unique()):
    cls_df = (
        df_attacks[df_attacks[LABEL_COL] == cls]
        .sample(frac=1, random_state=RANDOM_STATE)
        .reset_index(drop=True)
    )
    n       = len(cls_df)
    n_train = int(np.floor(n * TRAIN_FRAC))   # 50%
    n_cv    = int(np.floor(n * CV_FRAC))       # 25%
    n_test  = n - n_train - n_cv               # remainder (~25%)

    attack_train_parts.append(cls_df.iloc[:n_train])
    attack_cv_parts.append(   cls_df.iloc[n_train : n_train + n_cv])
    attack_test_parts.append( cls_df.iloc[n_train + n_cv :])

    split_log.append(dict(cls=cls, total=n,
                          train=n_train, crossval=n_cv, test=n_test))

attacks_train = pd.concat(attack_train_parts, ignore_index=True)
attacks_cv    = pd.concat(attack_cv_parts,    ignore_index=True)
attacks_test  = pd.concat(attack_test_parts,  ignore_index=True)

log_df = pd.DataFrame(split_log).set_index('cls')
log_df.index.name = LABEL_COL
log_df.loc['TOTAL'] = log_df.sum()
display(Markdown('### Per-class attack split counts'))
display(log_df)

### Per-class attack split counts

,total,train,crossval,test
Label,,,,
Bot,96154,48077,24038,24039
DDoS HOIC,1074379,537189,268594,268596
DDoS LOIC-HTTP,289328,144664,72332,72332
DDoS LOIC-UDP,2382,1191,595,596
DoS GoldenEye,26861,13430,6715,6716
DoS Hulk,1802966,901483,450741,450742
DoS Slowhttptest,105550,52775,26387,26388
DoS Slowloris,10274,5137,2568,2569
FTP-Patator,190300,95150,47575,47575


## 5. Step 4 — Benign completion without replacement

Each split is topped up with benign flows so that:

> `n_benign_in_split` = `n_attack_in_split`

This makes each split **50/50 balanced** between normal and attack traffic overall,  
while remaining imbalanced across attack types — the Rosay et al. design.  
All benign sampling is across the three splits **without replacement**.

In [ ]:
n_b_train = len(attacks_train)
n_b_cv    = len(attacks_cv)
n_b_test  = len(attacks_test)
n_b_total = n_b_train + n_b_cv + n_b_test

print('Attack flows assigned:')
print(f'  train_set    : {len(attacks_train):>10,}')
print(f'  crossval_set : {len(attacks_cv):>10,}')
print(f'  test_set     : {len(attacks_test):>10,}')
print(f'  total benign needed: {n_b_total:,}  (available: {len(df_benign):,})')

if n_b_total > len(df_benign):
    # Scale proportionally if benign pool is insufficient
    scale   = len(df_benign) / n_b_total
    n_b_train = int(n_b_train * scale)
    n_b_cv    = int(n_b_cv    * scale)
    n_b_test  = len(df_benign) - n_b_train - n_b_cv
    print(f'\n[WARN] Benign pool insufficient — scaled to {scale:.2%}.')

# Shuffle benign once, then slice without replacement
df_benign_shuf = (
    df_benign.sample(frac=1, random_state=RANDOM_STATE)
             .reset_index(drop=True)
)
benign_train = df_benign_shuf.iloc[:n_b_train]
benign_cv    = df_benign_shuf.iloc[n_b_train : n_b_train + n_b_cv]
benign_test  = df_benign_shuf.iloc[n_b_train + n_b_cv :
                                    n_b_train + n_b_cv + n_b_test]

print('\nBenign flows assigned:')
print(f'  train_set    : {len(benign_train):>10,}')
print(f'  crossval_set : {len(benign_cv):>10,}')
print(f'  test_set     : {len(benign_test):>10,}')
print(f'  benign unused: '
      f'{len(df_benign) - n_b_train - n_b_cv - n_b_test:,}')

Attack flows assigned:
  train_set    :  1,845,633
  crossval_set :    922,813
  test_set     :    922,822
  total benign needed: 3,691,268  (available: 10,000,000)

Benign flows assigned:
  train_set    :  1,845,633
  crossval_set :    922,813
  test_set     :    922,822
  benign unused: 6,308,732


## 6. Step 5 — Assemble and shuffle final splits

In [ ]:
def assemble(attacks, benign, seed):
    return (
        pd.concat([attacks, benign], ignore_index=True)
          .sample(frac=1, random_state=seed)
          .reset_index(drop=True)
    )

train_df = assemble(attacks_train, benign_train, RANDOM_STATE)
cv_df    = assemble(attacks_cv,    benign_cv,    RANDOM_STATE)
test_df  = assemble(attacks_test,  benign_test,  RANDOM_STATE)

print(f'train_set    : {len(train_df):>10,} rows  '
      f'(attacks: {len(attacks_train):,}  |  benign: {len(benign_train):,})')
print(f'crossval_set : {len(cv_df):>10,} rows  '
      f'(attacks: {len(attacks_cv):,}  |  benign: {len(benign_cv):,})')
print(f'test_set     : {len(test_df):>10,} rows  '
      f'(attacks: {len(attacks_test):,}  |  benign: {len(benign_test):,})')
print(f'\ncrossval vs test size diff: '
      f'{abs(len(cv_df) - len(test_df))} rows  '
      f'(≈0 expected; small diff from floor rounding on tiny classes)')

train_set    :  3,691,266 rows  (attacks: 1,845,633  |  benign: 1,845,633)
crossval_set :  1,845,626 rows  (attacks: 922,813  |  benign: 922,813)
test_set     :  1,845,644 rows  (attacks: 922,822  |  benign: 922,822)

crossval vs test size diff: 18 rows  (≈0 expected; small diff from floor rounding on tiny classes)


## 7. Step 6 — Verify class distribution in each split

In [ ]:
for name, df_s in [('train_set', train_df),
                   ('crossval_set', cv_df),
                   ('test_set', test_df)]:
    counts = df_s[LABEL_COL].value_counts().sort_values(ascending=False)
    pct    = (counts / len(df_s) * 100).round(3)
    display(Markdown(
        f'**{name}** — {len(df_s):,} total rows, '
        f'{df_s[LABEL_COL].nunique()} classes'
    ))
    display(pd.DataFrame({'count': counts, 'percent_%': pct})
              .rename_axis(LABEL_COL))

**train_set** — 3,691,266 total rows, 14 classes

,count,percent_%
Label,,
Benign,1845633,50.0000
DoS Hulk,901483,24.4220
DDoS HOIC,537189,14.5530
DDoS LOIC-HTTP,144664,3.9190
FTP-Patator,95150,2.5780
DoS Slowhttptest,52775,1.4300
Bot,48077,1.3020
SSH-Patator,46324,1.2550
DoS GoldenEye,13430,0.3640


**crossval_set** — 1,845,626 total rows, 14 classes

,count,percent_%
Label,,
Benign,922813,50.0000
DoS Hulk,450741,24.4220
DDoS HOIC,268594,14.5530
DDoS LOIC-HTTP,72332,3.9190
FTP-Patator,47575,2.5780
DoS Slowhttptest,26387,1.4300
Bot,24038,1.3020
SSH-Patator,23162,1.2550
DoS GoldenEye,6715,0.3640


**test_set** — 1,845,644 total rows, 14 classes

,count,percent_%
Label,,
Benign,922822,50.0000
DoS Hulk,450742,24.4220
DDoS HOIC,268596,14.5530
DDoS LOIC-HTTP,72332,3.9190
FTP-Patator,47575,2.5780
DoS Slowhttptest,26388,1.4300
Bot,24039,1.3020
SSH-Patator,23162,1.2550
DoS GoldenEye,6716,0.3640


## 8. Step 7 — Save splits to Parquet

In [ ]:
for split_name, split_df in [('train_set',    train_df),
                              ('crossval_set', cv_df),
                              ('test_set',     test_df)]:
    out_path = os.path.join(OUT_DIR, f'{split_name}.parquet')
    split_df.to_parquet(out_path, index=False, compression='snappy')
    size_mb = os.path.getsize(out_path) / 1e6
    print(f'  Saved  {split_name:15s}  →  {out_path}  ({size_mb:.1f} MB)')

print('\nAll splits saved.')

  Saved  train_set        →  data/lycos_unicas_ids2018\train_set.parquet  (428.6 MB)
  Saved  crossval_set     →  data/lycos_unicas_ids2018\crossval_set.parquet  (214.5 MB)
  Saved  test_set         →  data/lycos_unicas_ids2018\test_set.parquet  (215.0 MB)

All splits saved.


## 9. Step 8 — Compare against LycoS-IDS2017 splits

In [ ]:
LYCOS17_DIR = 'data/lycos-ids2017'   # adjust if needed

rows = []
for split_name in ['train_set', 'crossval_set', 'test_set']:
    for ds_label, ds_dir in [
        ('LycoS-Unicas-IDS2018', OUT_DIR),
        ('LycoS-IDS2017',        LYCOS17_DIR),
    ]:
        path = os.path.join(ds_dir, f'{split_name}.parquet')
        if not os.path.exists(path):
            rows.append({'dataset': ds_label, 'split': split_name,
                         'rows': 'NOT FOUND', 'features': '-',
                         'classes': '-', 'benign_%': '-'})
            continue
        df_s = pd.read_parquet(path)
        lc   = LABEL_COL if LABEL_COL in df_s.columns else df_s.columns[-1]
        n_b  = df_s[lc].astype(str).str.upper().isin(
                   ['BENIGN', '0', 'NORMAL']).sum()
        rows.append(dict(dataset=ds_label, split=split_name,
                         rows=f'{len(df_s):,}',
                         features=df_s.shape[1]-1,
                         classes=df_s[lc].nunique(),
                         **{'benign_%': f'{n_b/len(df_s)*100:.1f}%'}))

display(Markdown('### Split comparison — LycoS-Unicas-IDS2018 vs LycoS-IDS2017'))
display(pd.DataFrame(rows).sort_values(['split','dataset']))

# Feature schema diff
p18 = os.path.join(OUT_DIR,     'train_set.parquet')
p17 = os.path.join(LYCOS17_DIR, 'train_set.parquet')
if os.path.exists(p18) and os.path.exists(p17):
    c18 = set(pd.read_parquet(p18).columns) - {LABEL_COL}
    c17 = set(pd.read_parquet(p17).columns) - {LABEL_COL}
    print(f'\nShared features : {len(c18 & c17)}')
    o18 = sorted(c18 - c17)
    o17 = sorted(c17 - c18)
    if o18: print(f'Only in LycoS18 : {o18}')
    if o17: print(f'Only in LycoS17 : {o17}')
    if not o18 and not o17:
        print('Feature sets are identical — same LycoSTand schema.')
else:
    print('LycoS-IDS2017 splits not found — update LYCOS17_DIR to enable.')

### Split comparison — LycoS-Unicas-IDS2018 vs LycoS-IDS2017

,dataset,split,rows,features,classes,benign_%
3,LycoS-IDS2017,crossval_set,"220,312",82,14,50.0%
2,LycoS-Unicas-IDS2018,crossval_set,"1,845,626",77,14,50.0%
5,LycoS-IDS2017,test_set,"220,312",82,14,50.0%
4,LycoS-Unicas-IDS2018,test_set,"1,845,644",77,14,50.0%
1,LycoS-IDS2017,train_set,"440,632",82,14,50.0%
0,LycoS-Unicas-IDS2018,train_set,"3,691,266",77,14,50.0%



Shared features : 77
Only in LycoS17 : ['dst_addr', 'flow_id', 'label', 'src_addr', 'src_port', 'timestamp']


## 10. Final file listing

In [ ]:
display(Markdown(f'### Files written to `{OUT_DIR}/`'))
for fname in sorted(os.listdir(OUT_DIR)):
    fpath = os.path.join(OUT_DIR, fname)
    print(f'  {fname:<35s}  {os.path.getsize(fpath)/1e6:>8.1f} MB')

print()
print('Load any split:')
print("  pd.read_parquet('data/lycos_unicas_ids2018/train_set.parquet')")
print("  pd.read_parquet('data/lycos_unicas_ids2018/crossval_set.parquet')")
print("  pd.read_parquet('data/lycos_unicas_ids2018/test_set.parquet')")
print()
print('Rosay et al. training protocol:')
print('  train_set + crossval_set → training & hyperparameter tuning')
print('  test_set                 → final scoring only (never seen during training)')

### Files written to `data/lycos_unicas_ids2018/`

  _lycos2018_clean.parquet               1409.3 MB
  crossval_set.parquet                    214.5 MB
  test_set.parquet                        215.0 MB
  train_set.parquet                       428.6 MB

Load any split:
  pd.read_parquet('data/lycos_unicas_ids2018/train_set.parquet')
  pd.read_parquet('data/lycos_unicas_ids2018/crossval_set.parquet')
  pd.read_parquet('data/lycos_unicas_ids2018/test_set.parquet')

Rosay et al. training protocol:
  train_set + crossval_set → training & hyperparameter tuning
  test_set                 → final scoring only (never seen during training)
